In [1]:
!pip -q install google-genai

In [2]:
from google import genai
from google.colab import files
from getpass import getpass
import time

api_key = getpass("Enter your Gemini API Key: ")
client = genai.Client(api_key=api_key)

Enter your Gemini API Key: ··········


In [3]:
print("AI RESEARCH PAPER SUMMARIZER & REVIEW GENERATOR")
print("=" * 55)

print("\nUpload your research paper (PDF only):")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"\nUploaded: {pdf_filename}")

# upload to gemini file api
print("Sending to Gemini File API...")
paper_file = client.files.upload(
    file=pdf_filename,
    config={"mime_type": "application/pdf"}
)

# wait until gemini has fully processed the file
while paper_file.state.name == "PROCESSING":
    time.sleep(2)
    paper_file = client.files.get(name=paper_file.name)

if paper_file.state.name == "FAILED":
    raise ValueError("File processing failed. Please try uploading again.")

print("File ready.\n")

print("Select difficulty level:")
print("  1. Beginner")
print("  2. Technical")
print("  3. Literature Review")
level_choice = input("Enter 1 / 2 / 3: ").strip()

level_map = {
    "1": "Beginner — explain in simple, clear language suitable for someone new to this field. Avoid heavy jargon and define any technical terms used.",
    "2": "Technical — use domain-specific terminology and assume the reader has strong technical knowledge of the subject area.",
    "3": "Literature Review — frame the entire analysis in the context of existing research. Highlight how this paper contributes to, challenges, or extends prior work in the field."
}
difficulty = level_map.get(level_choice, level_map["2"])

AI RESEARCH PAPER SUMMARIZER & REVIEW GENERATOR

Upload your research paper (PDF only):


Saving research_paper.pdf to research_paper.pdf

Uploaded: research_paper.pdf
Sending to Gemini File API...
File ready.

Select difficulty level:
  1. Beginner
  2. Technical
  3. Literature Review
Enter 1 / 2 / 3: 3


In [4]:
def call_gemini(prompt_text):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            {
                "role": "user",
                "parts": [
                    {"file_data": {"mime_type": "application/pdf", "file_uri": paper_file.uri}},
                    {"text": prompt_text}
                ]
            }
        ]
    )
    return response.text


# --- Prompt 1: Structured Summary ---
prompt_summary = f"""
You are an expert academic researcher and scientific writer with deep experience analyzing research papers across all domains.

Read the entire research paper provided and produce a structured, well-organized summary.

Difficulty Level: {difficulty}

Generate the output strictly in this format with no deviations:

STRUCTURED SUMMARY
------------------
Overview:
[2-3 sentence overview of what this paper is about and why it matters]

Problem Statement:
[What specific problem or gap does this paper address?]

Key Contributions:
• [Contribution 1]
• [Contribution 2]
• [Contribution 3]

Methodology:
[Describe the approach, techniques, models, or methods used in detail]

Dataset / Experimental Setup:
[Describe datasets used, benchmarks, evaluation metrics, or experimental conditions]

Results & Findings:
[Key quantitative or qualitative results and what they indicate about the research]

Rules:
- Read the full paper including figures, tables, and references before responding.
- Match language style and depth strictly to the difficulty level above.
- Be factual and grounded only in what the paper contains — do not infer or fabricate.
- If any section is genuinely not covered in the paper, write "Not explicitly mentioned".
- Do not add any extra sections or commentary outside the format above.
"""


# --- Prompt 2: Peer Review ---
prompt_review = f"""
You are a senior peer reviewer for a top-tier academic journal such as Nature, IEEE, or ACM. You are known for balanced, precise, and constructive evaluations that help authors improve their work.

Read the entire research paper provided and produce a formal peer review.

Difficulty Level: {difficulty}

Generate the output strictly in this format with no deviations:

PEER REVIEW
-----------
Strengths:
• [Strength 1 — be specific, reference what the paper actually does well]
• [Strength 2]
• [Strength 3]

Weaknesses:
• [Weakness 1 — constructive and actionable, not dismissive]
• [Weakness 2]
• [Weakness 3]

Reviewer Comments:
[3-4 sentences of overall impression, specific suggestions for improvement, and what would make this paper stronger]

Overall Score: [X / 10]
Recommendation: [Accept / Minor Revision / Major Revision / Reject]

Rules:
- Score must honestly reflect novelty, clarity, methodology rigor, and quality of results.
- Every weakness must be paired with an implicit or explicit suggestion for how to fix it.
- Strengths must be specific to this paper — no generic praise.
- Match depth and language to the difficulty level: {difficulty}
- Base the review entirely on the paper content. Do not fabricate claims.
"""


# --- Prompt 3: Key Terms Extraction ---
prompt_terms = f"""
You are a scientific knowledge extraction specialist. Your expertise is in identifying the most important technical and conceptual terms from research papers and explaining them clearly.

Read the entire research paper provided and extract the most important terms and concepts.

Difficulty Level: {difficulty}

Generate the output strictly in this format with no deviations:

KEY TERMS & CONCEPTS
--------------------
• [Term 1]: [One clear sentence explaining this term in context of the paper]
• [Term 2]: [One clear sentence]
• [Term 3]: [One clear sentence]
• [Term 4]: [One clear sentence]
• [Term 5]: [One clear sentence]
• [Term 6]: [One clear sentence]
• [Term 7]: [One clear sentence]
• [Term 8]: [One clear sentence]

Rules:
- Extract exactly 8 terms. No more, no less.
- Prioritize: core technical concepts, model or algorithm names, datasets, evaluation metrics, and domain-specific terminology.
- Each explanation must be one sentence only — concise and directly tied to how the term is used in this paper.
- Match explanation language strictly to the difficulty level: {difficulty}
- Do not pick generic words — every term must be meaningful and specific to this paper.
"""


# --- Prompt 4: Future Research Directions ---
prompt_future = f"""
You are a research strategist and senior academic advisor with expertise in identifying gaps in current literature and proposing high-impact future research directions.

Read the entire research paper provided and identify research gaps and future directions.

Difficulty Level: {difficulty}

Generate the output strictly in this format with no deviations:

FUTURE RESEARCH DIRECTIONS
--------------------------
Identified Gaps:
• [Gap 1 — what this paper leaves unanswered, unexplored, or explicitly acknowledges as a limitation]
• [Gap 2]
• [Gap 3]

Suggested Future Work:
• [Direction 1 — specific, feasible, directly grounded in this paper's context]
• [Direction 2]
• [Direction 3]
• [Direction 4]

Potential Real-World Applications:
• [Application 1 — concrete real-world use case that could emerge from this research]
• [Application 2]
• [Application 3]

Rules:
- All gaps and directions must be directly grounded in this specific paper — no generic suggestions.
- Avoid vague statements like "more research is needed" — be specific about what, how, and why.
- Real-world applications must be practical and realistic, not speculative.
- Match depth and language to difficulty level: {difficulty}
"""


# --- Prompt 5: Quick Summary ---
prompt_quick = f"""
You are a science communicator who specializes in making complex research accessible, engaging, and clear for different audiences.

Read the entire research paper provided and write a quick summary.

Difficulty Level: {difficulty}

Generate the output strictly in this format with no deviations:

QUICK SUMMARY (30-second read)
------------------------------
[Write exactly one paragraph of 5-7 sentences. Cover: what problem is being solved, how it is approached, what was achieved, and why it matters. The paragraph must read naturally and feel engaging — not like a list compressed into sentences. Do not start with "This paper". Make the opening line draw the reader in.]

Rules:
- Exactly one paragraph. No bullet points. No sub-headings inside the paragraph.
- Must be readable comfortably in under 30 seconds.
- Tone and vocabulary must match strictly to difficulty level: {difficulty}
- The opening sentence must be engaging and original — not a generic academic opener.
- Accuracy is non-negotiable — every claim must be grounded in the paper.
"""


prompts = [
    ("Structured Summary", prompt_summary),
    ("Peer Review", prompt_review),
    ("Key Terms & Concepts", prompt_terms),
    ("Future Research Directions", prompt_future),
    ("Quick Summary", prompt_quick),
]

results = {}
print("\nAnalyzing paper... please wait.\n")
for label, prompt in prompts:
    print(f"  Generating {label}...")
    results[label] = call_gemini(prompt)
    time.sleep(1)

print("\nAnalysis complete.")


Analyzing paper... please wait.

  Generating Structured Summary...
  Generating Peer Review...
  Generating Key Terms & Concepts...
  Generating Future Research Directions...
  Generating Quick Summary...

Analysis complete.


In [5]:
print("\n" + "=" * 55)
print("         FULL ANALYSIS REPORT")
print("=" * 55)

for label, output in results.items():
    print(f"\n{'=' * 55}")
    print(output)

print("\n" + "=" * 55)
print("END OF REPORT")
print("=" * 55)

# delete the uploaded file from gemini after use
client.files.delete(name=paper_file.name)


         FULL ANALYSIS REPORT

STRUCTURED SUMMARY
------------------
Overview:
This paper introduces a novel deep learning-based model, aADGA (adaptive auto-encoder with genetic algorithm), for early epilepsy seizure prediction using Electroencephalogram (EEG) recordings. It aims to improve prediction accuracy and reduce false alarm rates, significantly impacting the quality of life for epilepsy patients by enabling timely intervention. The approach addresses the challenges of computational complexity and classification accuracy in existing seizure prediction methods.

Problem Statement:
The paper addresses the challenge of accurately and efficiently predicting epileptic seizures in advance. Existing methods often suffer from computational complexity, lower classification accuracy, high false alarm rates, and inadequate performance in real-time applications. There is a need for an intelligent learning model that can optimize feature learning without extensive pre-processing and achiev

DeleteFileResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=10>
  )
)

In [7]:
!pip -q install gradio pypdf

In [12]:
!pip -q install gradio

import time
import gradio as gr


def upload_pdf(client, pdf_path):
    paper_file = client.files.upload(
        file=pdf_path,
        config={"mime_type": "application/pdf"}
    )
    while paper_file.state.name == "PROCESSING":
        time.sleep(2)
        paper_file = client.files.get(name=paper_file.name)
    if paper_file.state.name == "FAILED":
        raise ValueError("File processing failed. Please try again.")
    return paper_file


def call_gemini(client, file_uri, prompt_text):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            {
                "role": "user",
                "parts": [
                    {"file_data": {"mime_type": "application/pdf", "file_uri": file_uri}},
                    {"text": prompt_text}
                ]
            }
        ]
    )
    return response.text


def build_prompts(difficulty):
    prompt_summary = f"""
You are an expert academic researcher and scientific writer with deep experience analyzing research papers across all domains.
Read the entire research paper provided and produce a structured, well-organized summary.
Difficulty Level: {difficulty}

Generate the output using clean Markdown formatting as shown below. Use ## for section headers, ** for bold labels, and - for bullet points.

## Structured Summary

**Overview**
[2-3 sentence overview of what this paper is about and why it matters]

**Problem Statement**
[What specific problem or gap does this paper address?]

**Key Contributions**
- [Contribution 1]
- [Contribution 2]
- [Contribution 3]

**Methodology**
[Describe the approach, techniques, models, or methods used]

**Dataset / Experimental Setup**
[Describe datasets used, benchmarks, evaluation metrics, or experimental conditions]

**Results & Findings**
[Key quantitative or qualitative results and what they indicate]

Rules:
- Read the full paper including figures and tables before responding.
- Match language style and depth strictly to the difficulty level.
- Be factual — do not infer or fabricate anything not in the paper.
- If a section is not covered in the paper, write "Not explicitly mentioned".
- Do not add sections outside the format above.
- Output must be valid Markdown only — no plain text headers.
"""

    prompt_review = f"""
You are a senior peer reviewer for a top-tier academic journal such as Nature, IEEE, or ACM.
Read the entire research paper and produce a formal peer review.
Difficulty Level: {difficulty}

Generate the output using clean Markdown formatting. Use ## for section headers, ** for bold labels, and - for bullet points.

## Peer Review

**Strengths**
- [Strength 1 — specific to what this paper actually does well]
- [Strength 2]
- [Strength 3]

**Weaknesses**
- [Weakness 1 — constructive and actionable]
- [Weakness 2]
- [Weakness 3]

**Reviewer Comments**
[3-4 sentences of overall impression and specific improvement suggestions]

**Overall Score:** [X / 10]
**Recommendation:** [Accept / Minor Revision / Major Revision / Reject]

Rules:
- Score must honestly reflect novelty, clarity, methodology rigor, and results quality.
- Every weakness must suggest how to fix it directly or implicitly.
- No generic praise — strengths must be specific to this paper.
- Match language and depth to difficulty level: {difficulty}
- Output must be valid Markdown only.
"""

    prompt_terms = f"""
You are a scientific knowledge extraction specialist.
Read the entire research paper and extract the most important terms and concepts.
Difficulty Level: {difficulty}

Generate the output using clean Markdown formatting.

## Key Terms & Concepts

| Term | Explanation |
|------|-------------|
| [Term 1] | [One clear sentence in context of the paper] |
| [Term 2] | [One clear sentence] |
| [Term 3] | [One clear sentence] |
| [Term 4] | [One clear sentence] |
| [Term 5] | [One clear sentence] |
| [Term 6] | [One clear sentence] |
| [Term 7] | [One clear sentence] |
| [Term 8] | [One clear sentence] |

Rules:
- Extract exactly 8 terms. No more, no less.
- Prioritize technical concepts, model names, datasets, and evaluation metrics.
- One sentence per term — concise and tied to how the term is used in this paper.
- Match explanation language to difficulty level: {difficulty}
- Output must be a valid Markdown table only.
"""

    prompt_future = f"""
You are a research strategist and senior academic advisor.
Read the entire research paper and identify research gaps and future directions.
Difficulty Level: {difficulty}

Generate the output using clean Markdown formatting. Use ## for section headers and - for bullet points.

## Future Research Directions

**Identified Gaps**
- [Gap 1 — what this paper leaves unanswered or explicitly acknowledges as a limitation]
- [Gap 2]
- [Gap 3]

**Suggested Future Work**
- [Direction 1 — specific and grounded in this paper's context]
- [Direction 2]
- [Direction 3]
- [Direction 4]

**Potential Real-World Applications**
- [Application 1 — concrete and realistic]
- [Application 2]
- [Application 3]

Rules:
- All suggestions must be grounded in this specific paper — no generic research ideas.
- Avoid vague statements — be specific about what, how, and why.
- Match depth and language to difficulty level: {difficulty}
- Output must be valid Markdown only.
"""

    prompt_quick = f"""
You are a science communicator who makes complex research accessible and engaging.
Read the entire research paper and write a quick summary.
Difficulty Level: {difficulty}

Generate the output using clean Markdown formatting.

## Quick Summary ⚡
*30-second read*

[Write exactly one paragraph of 5-7 sentences. Cover: what problem is solved, how it is approached, what was achieved, and why it matters. Do not start with "This paper". Make the opening sentence draw the reader in. No bullet points inside the paragraph.]

Rules:
- Exactly one paragraph. No sub-headings or bullet points inside.
- Readable in under 30 seconds.
- Tone and vocabulary must match difficulty level: {difficulty}
- Every claim must be grounded in the paper.
- Output must be valid Markdown only.
"""

    return [
        ("📄 Structured Summary", prompt_summary),
        ("🧪 Peer Review", prompt_review),
        ("🔑 Key Terms & Concepts", prompt_terms),
        ("🔭 Future Research", prompt_future),
        ("⚡ Quick Summary", prompt_quick),
    ]


css = """
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
    max-width: 1200px !important;
    margin: auto;
}
#header {
    text-align: center;
    padding: 24px 0 8px 0;
}
#header h1 {
    font-size: 2rem;
    font-weight: 700;
    color: #1a1a2e;
    margin-bottom: 6px;
}
#header p {
    color: #555;
    font-size: 1rem;
}
#analyze_btn {
    background: linear-gradient(135deg, #667eea, #764ba2) !important;
    color: white !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
    border-radius: 10px !important;
    padding: 12px !important;
    border: none !important;
    cursor: pointer !important;
    width: 100% !important;
    margin-top: 8px !important;
}
#analyze_btn:hover {
    opacity: 0.9 !important;
}
.output-markdown {
    font-family: 'Segoe UI', sans-serif !important;
    font-size: 0.93rem !important;
    line-height: 1.8 !important;
    color: #1a1a2e !important;
    padding: 16px !important;
}
.output-markdown h2 {
    font-size: 1.2rem !important;
    font-weight: 700 !important;
    color: #4a4a8a !important;
    border-bottom: 2px solid #e0e0f0 !important;
    padding-bottom: 6px !important;
    margin-top: 8px !important;
}
.output-markdown strong {
    color: #333 !important;
}
.output-markdown table {
    width: 100% !important;
    border-collapse: collapse !important;
    margin-top: 8px !important;
}
.output-markdown th {
    background: #f0f0fa !important;
    padding: 8px 12px !important;
    text-align: left !important;
    font-weight: 600 !important;
    border: 1px solid #ddd !important;
}
.output-markdown td {
    padding: 8px 12px !important;
    border: 1px solid #ddd !important;
    vertical-align: top !important;
}
.output-markdown tr:nth-child(even) {
    background: #fafafa !important;
}
"""


def analyze_paper_ui(pdf_file, difficulty_label, progress=gr.Progress()):
    if pdf_file is None:
        return ["Please upload a research paper PDF."] * 5

    level_map = {
        "Beginner":          "Beginner — explain in simple, clear language suitable for someone new to this field.",
        "Technical":         "Technical — use domain-specific terminology and assume strong technical knowledge.",
        "Literature Review": "Literature Review — frame the analysis in the context of existing research and highlight how this paper contributes to the broader field."
    }
    difficulty = level_map.get(difficulty_label, level_map["Technical"])

    try:
        progress(0.05, desc="Uploading PDF to Gemini File API...")
        paper_file = upload_pdf(client, pdf_file.name)

        prompts = build_prompts(difficulty)
        results = []
        steps = len(prompts)

        for i, (label, prompt) in enumerate(prompts):
            progress((i + 1) / (steps + 1), desc=f"Generating {label}...")
            output = call_gemini(client, paper_file.uri, prompt)
            # strip markdown code fences if model wraps output in them
            output = output.strip().removeprefix("```markdown").removeprefix("```").removesuffix("```").strip()
            results.append(output)
            time.sleep(0.5)

        progress(1.0, desc="Done!")
        client.files.delete(name=paper_file.name)
        return results

    except Exception as e:
        return [f"**Error:** {str(e)}"] * 5


with gr.Blocks(css=css, title="AI Research Paper Analyzer") as app:

    gr.HTML("""
        <div id="header">
            <h1>🔬 AI Research Paper Analyzer</h1>
            <p>Upload any research paper PDF and get an instant structured analysis powered by Gemini</p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            pdf_input = gr.File(
                label="Upload Research Paper (PDF)",
                file_types=[".pdf"]
            )
            difficulty_input = gr.Radio(
                choices=["Beginner", "Technical", "Literature Review"],
                value="Technical",
                label="Analysis Depth"
            )
            analyze_btn = gr.Button("Analyze Paper", elem_id="analyze_btn")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📄 Structured Summary"):
                    out_summary = gr.Markdown(
                        value="*Your structured summary will appear here...*",
                        elem_classes=["output-markdown"]
                    )
                with gr.Tab("🧪 Peer Review"):
                    out_review = gr.Markdown(
                        value="*Peer review analysis will appear here...*",
                        elem_classes=["output-markdown"]
                    )
                with gr.Tab("🔑 Key Terms"):
                    out_terms = gr.Markdown(
                        value="*Key terms and concepts will appear here...*",
                        elem_classes=["output-markdown"]
                    )
                with gr.Tab("🔭 Future Research"):
                    out_future = gr.Markdown(
                        value="*Future research directions will appear here...*",
                        elem_classes=["output-markdown"]
                    )
                with gr.Tab("⚡ Quick Summary"):
                    out_quick = gr.Markdown(
                        value="*Your 30-second summary will appear here...*",
                        elem_classes=["output-markdown"]
                    )

    analyze_btn.click(
        fn=analyze_paper_ui,
        inputs=[pdf_input, difficulty_input],
        outputs=[out_summary, out_review, out_terms, out_future, out_quick]
    )


app.launch(share=True)

/tmp/ipykernel_342/2513127503.py:308: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="AI Research Paper Analyzer") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6c9ac9acdf3b5942a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
